# Cross-Validation and Model Selection — Exercises

Companion to the note [Cross-Validation and Model Selection](Cross-Validation%20and%20Model%20Selection.md).

Practise train/validation/test discipline, k-fold and its variants (stratified, leave-one-out, group, time-series), nested CV, hyperparameter search budgets, information criteria (AIC/BIC) and leak-free pipelines, implementing the splitters and searches from scratch.

**18 exercises** · 🧠 Concept ×5 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import (KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit, cross_val_score, GridSearchCV)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy import stats
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Three splits, three jobs
*🧠 Concept · ★☆☆*

Explain what each of train, validation and test is used for. A student tunes the regularization strength on the test set and reports the test
score. Why is the reported number biased, and in which direction? What would k-fold CV replace in this setup?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Any data used to make a decision about the model can no longer give an unbiased estimate of that model's performance.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Train** fits parameters ($w$), **validation** chooses hyperparameters ($\lambda$, degree, architecture), **test** estimates the final generalization error once.
Tuning on test makes it a validation set: the chosen $\lambda$ is the one that happened to fit *this* test sample best, so the score is **optimistically** biased.
k-fold CV replaces the single validation split (on the training data), using all of it for both fitting and validating and reducing variance.

</details>

### Exercise 2 · Pick the splitter
*🧠 Concept · ★☆☆*

Which CV scheme from the note fits each case? (a) 40 patients, 20 ECG recordings each. (b) Daily sales for 3 years. (c) Fraud with 1 % positives.
(d) 25 labelled samples in total. (e) Hourly sensor data from 10 machines, where the model will be deployed on new machines.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does the deployment situation look like? The validation split should mimic it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Group k-fold** by patient (new patients at deployment). (b) **Time-series split** (train on the past, validate on the future; never shuffle).
(c) **Stratified** k-fold so each fold has positives. (d) **Leave-one-out** or repeated (stratified) k-fold: every sample is precious.
(e) Group k-fold by machine, combined with temporal ordering if the future matters too.

</details>

### Exercise 3 · Why nested CV?
*🧠 Concept · ★★☆*

`GridSearchCV(...).best_score_` is the CV score of the best hyperparameter setting. (a) Why is it an optimistic estimate of the tuned model's performance?
(b) Describe nested CV. (c) What exactly does the nested score estimate: the performance of one specific model, or of a procedure?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The best of many noisy estimates is biased upward, even if every single estimate is unbiased.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) We pick the maximum of several noisy CV estimates; $\mathbb E[\max_j \hat s_j] \ge \max_j \mathbb E[\hat s_j]$ (the winner's curse), so the same folds that
chose the setting cannot evaluate it fairly.
(b) **Outer loop**: k folds; for each, run the full tuning procedure (inner CV grid search) on the outer-training part only, refit, then score on the outer
test fold. Average the outer scores.
(c) It estimates the generalization of the **whole procedure** "tune by inner CV, then refit", not of one particular hyperparameter value. The final deployed
model is obtained by running the procedure once on all data.

</details>

### Exercise 4 · Grid vs. random search
*🧠 Concept · ★★☆*

With a budget of 9 trials and 2 hyperparameters, of which only one actually matters, compare a 3×3 grid with 9 random points.
How many distinct values of the important hyperparameter does each try? Why is Bayesian optimisation or successive halving better still?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Project the 9 points onto the axis of the important hyperparameter.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The grid tries only **3** distinct values of the important parameter (each 3 times); random search tries **9** distinct values (Bergstra & Bengio, 2012).
Since many problems have low effective dimensionality, random search usually wins for the same budget. Bayesian optimisation uses past results to pick
promising points; successive halving / Hyperband stop bad configurations early, so more configurations fit in the budget.

</details>

### Exercise 5 · AIC, BIC and CV
*🧠 Concept · ★★☆*

Compare AIC $=2k-2\ln\hat L$ and BIC $=k\ln N-2\ln\hat L$. For which $N$ does BIC penalise each parameter more than AIC? Which criterion tends to pick
larger models? When would you prefer information criteria over cross-validation, and when not?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare the per-parameter penalties $2$ and $\ln N$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

BIC penalises more when $\ln N > 2$, i.e. $N > e^2\approx7.4$, so practically always; AIC tends to choose **larger** models (good for prediction),
BIC smaller ones (consistent: it finds the true model as $N\to\infty$ if it is in the candidate set).
Information criteria need only one fit per model and no held-out data, but require a likelihood and a meaningful parameter count $k$ (unclear for
regularized models, trees, neural nets). CV is model-agnostic and directly estimates predictive error, at $k$ times the cost.

</details>

## Part B · By hand

### Exercise 6 · Counting fits
*✍️ By hand · ★☆☆*

$N = 1000$ samples. (a) In 5-fold CV, how many samples are used for training and validation in each fold? (b) A grid search over 20 settings with 5-fold CV plus a final refit:
how many model fits? (c) Nested CV with 5 outer and 3 inner folds over the same 20 settings, with a refit on each outer-training set: how many fits?

In [ ]:
n_train_fold = None
n_val_fold = None
fits_grid = None
fits_nested = None

_tr, _va = next(KFold(5).split(np.zeros(1000)))
check('train size', n_train_fold, len(_tr)); check('val size', n_val_fold, len(_va))
check('grid fits', fits_grid, 20 * 5 + 1)
check('nested fits', fits_nested, sum(20 * 3 + 1 for _ in range(5)))

<details>
<summary><b>💡 Hint</b></summary>

Each outer fold runs a complete inner grid search (including its refit).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) 800 train, 200 validation. (b) $20\cdot5 + 1 = 101$. (c) $5\cdot(20\cdot3 + 1) = 305$. Nested CV is expensive, which is why it is used for
*estimating* performance, not routinely for choosing the final model.

```python
n_train_fold = 800
n_val_fold = 200
fits_grid = 101
fits_nested = 305
```

</details>

### Exercise 7 · How correlated are the folds?
*✍️ By hand · ★★☆*

In $k$-fold CV, two different training sets share how many samples, as a fraction of the training-set size? Evaluate for $k=5$, $k=10$ and LOOCV with $N=100$.
Use this to explain why LOOCV has **low bias but high variance** as an estimator.

In [ ]:
overlap_5 = None
overlap_10 = None
overlap_loo = None

def _overlap(k, N=100):
    folds = list(KFold(k).split(np.zeros(N)))
    a, b = set(folds[0][0]), set(folds[1][0])
    return len(a & b) / len(a)
check('k=5', overlap_5, _overlap(5)); check('k=10', overlap_10, _overlap(10)); check('LOO', overlap_loo, _overlap(100))

<details>
<summary><b>💡 Hint</b></summary>

Two training sets each omit one (different) fold, so they share $k-2$ of the $k$ folds; each training set has $k-1$ folds.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Overlap $= \frac{k-2}{k-1}$: $3/4 = 0.75$ for $k=5$, $8/9\approx0.889$ for $k=10$, $98/99\approx0.990$ for LOO ($N=100$).
Each LOO model is trained on almost all data (low bias), but the $N$ models are nearly identical, so their errors are highly correlated: averaging
correlated quantities does not reduce variance much. $k=5$ or $10$ is a good compromise.

```python
overlap_5 = 3 / 4
overlap_10 = 8 / 9
overlap_loo = 98 / 99
```

</details>

### Exercise 8 · AIC and BIC disagree
*✍️ By hand · ★☆☆*

Two models are fitted on $N = 100$ samples. Model A: $k=3$, $\ln\hat L = -120$. Model B: $k=6$, $\ln\hat L = -115$.
Compute AIC and BIC for both. Which model does each criterion prefer?

In [ ]:
aic_A = None
aic_B = None
bic_A = None
bic_B = None

_aic = lambda k, ll: 2 * k - 2 * ll
_bic = lambda k, ll, n=100: k * np.log(n) - 2 * ll
check('AIC A', aic_A, _aic(3, -120)); check('AIC B', aic_B, _aic(6, -115))
check('BIC A', bic_A, _bic(3, -120)); check('BIC B', bic_B, _bic(6, -115))

<details>
<summary><b>💡 Hint</b></summary>

Lower is better for both. $\ln 100\approx4.605$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

AIC: A $= 6 + 240 = 246$, B $= 12 + 230 = 242$ → **B**. BIC: A $= 13.82 + 240 = 253.82$, B $= 27.63 + 230 = 257.63$ → **A**.
Three extra parameters buy 5 log-likelihood units: enough for AIC (penalty 2 each), not for BIC (penalty 4.6 each).

```python
aic_A, aic_B = 246.0, 242.0
bic_A = 3 * np.log(100) + 240
bic_B = 6 * np.log(100) + 230
```

</details>

### Exercise 9 · How many random trials?
*✍️ By hand · ★★☆*

Suppose 5 % of the hyperparameter space is "good" (within tolerance of the optimum). How many independent random-search trials are needed to hit the good region
with probability at least 95 %? Does the answer depend on the number of hyperparameters?

In [ ]:
n_trials = None

check('trials', n_trials, int(np.ceil(np.log(0.05) / np.log(0.95))))

<details>
<summary><b>💡 Hint</b></summary>

$P(\text{no hit in } n) = 0.95^n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$1 - 0.95^n \ge 0.95 \iff n \ge \ln0.05/\ln0.95 \approx 58.4$, so **59** trials. The bound depends only on the *volume fraction* of the good region,
not on the dimension (but in high dimensions the good region usually occupies a smaller fraction).

```python
n_trials = 59
```

</details>

### Exercise 10 · Successive halving budget
*✍️ By hand · ★★☆*

Successive halving with $n_0 = 81$ configurations, reduction factor $\eta = 3$, and minimum resource $r_0 = 1$ epoch: each round keeps the best $1/\eta$ of the
configurations and multiplies their resource by $\eta$, until one configuration remains. List the rounds, compute the total budget in epochs, and compare with
training all 81 configurations for 81 epochs.

In [ ]:
budget_sh = None
budget_full = None

_n, _r, _tot = 81, 1, 0
while _n >= 1:
    _tot += _n * _r
    if _n == 1: break
    _n, _r = _n // 3, _r * 3
check('successive halving', budget_sh, _tot); check('full training', budget_full, 81 * 81)

<details>
<summary><b>💡 Hint</b></summary>

Rounds: $81\times1,\ 27\times3,\ \dots$

</details>

<details>
<summary><b>✅ Solution</b></summary>

Rounds $81\times1,\ 27\times3,\ 9\times9,\ 3\times27,\ 1\times81$: each costs 81 epochs, total **405** versus **6561**, a 16× saving.
The risk: a configuration that learns slowly may be eliminated early; Hyperband hedges by running several brackets with different $r_0$.

```python
budget_sh = 405
budget_full = 6561
```

</details>

## Part C · Code

### Exercise 11 · k-fold indices from scratch
*💻 Code · ★☆☆*

Implement `kfold_indices(N, k)` returning a list of `(train_idx, val_idx)` pairs **without shuffling**, where the first `N % k` folds get one extra sample
(sklearn's convention). Verify it on $N=23$, $k=5$.

In [ ]:
def kfold_indices(N, k):
    return None

folds_23 = kfold_indices(23, 5)

_ref = list(KFold(5).split(np.zeros(23)))
check('validation folds', None if folds_23 is None else all(np.array_equal(a[1], b[1]) for a, b in zip(folds_23, _ref)), True)
check('training folds', None if folds_23 is None else all(np.array_equal(np.sort(a[0]), b[0]) for a, b in zip(folds_23, _ref)), True)

<details>
<summary><b>💡 Hint</b></summary>

Fold sizes: `np.full(k, N // k); sizes[:N % k] += 1`; then walk through `np.arange(N)` with a running start index.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Every sample is in exactly one validation fold; the train set is its complement.

```python
def kfold_indices(N, k):
    sizes = np.full(k, N // k); sizes[:N % k] += 1
    idx, out, start = np.arange(N), [], 0
    for s in sizes:
        val = idx[start:start + s]
        out.append((np.r_[idx[:start], idx[start + s:]], val))
        start += s
    return out

folds_23 = kfold_indices(23, 5)
```

</details>

### Exercise 12 · Stratified k-fold from scratch
*💻 Code · ★★☆*

Implement `stratified_folds(y, k, seed)` returning a list of `k` validation-index arrays such that each class is spread as evenly as possible:
shuffle each class's indices and deal them round-robin into the folds. Check on a 90/10 label vector that each fold has 1 or 2 positives
when $N=100$, $k=7$.

In [ ]:
y_imb = np.r_[np.zeros(90, int), np.ones(10, int)]

def stratified_folds(y, k, seed=0):
    return None

sfolds = stratified_folds(y_imb, 7)

if sfolds is not None:
    _all = np.sort(np.concatenate(sfolds))
    check('folds partition the data', np.array_equal(_all, np.arange(100)), True)
    check('class counts per fold differ by at most 1', all(np.ptp([np.sum(y_imb[f] == c) for f in sfolds]) <= 1 for c in (0, 1)), True)
else:
    check('stratified folds', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep a running fold counter across classes so the fold sizes also stay balanced: `fold = (offset + j) % k`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Plain k-fold on sorted or imbalanced labels can create folds with **no** positives, making recall undefined and estimates noisy. Stratification keeps
each fold's class ratio close to the overall one. (sklearn's `StratifiedKFold` uses a slightly different allocation but the same idea.)

```python
y_imb = np.r_[np.zeros(90, int), np.ones(10, int)]

def stratified_folds(y, k, seed=0):
    r = np.random.default_rng(seed)
    folds = [[] for _ in range(k)]
    offset = 0
    for c in np.unique(y):
        idx = r.permutation(np.flatnonzero(y == c))
        for j, i in enumerate(idx):
            folds[(offset + j) % k].append(i)
        offset += len(idx)
    return [np.array(sorted(f)) for f in folds]

sfolds = stratified_folds(y_imb, 7)
print([int(y_imb[f].sum()) for f in sfolds])
```

</details>

### Exercise 13 · Cross-validation with the scaler inside
*💻 Code · ★★☆*

Using `KFold(5, shuffle=True, random_state=0)` on breast cancer, implement `cv_score(X, y, splitter)` that, **in each fold**, standardises with the
training-fold statistics, fits `LogisticRegression(max_iter=1000)` and records accuracy. Compare with `cross_val_score` on a pipeline.

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
cv5 = KFold(5, shuffle=True, random_state=0)

def cv_score(X, y, splitter):
    return None   # list of fold accuracies

scores_manual = cv_score(X_bc, y_bc, cv5)

check('fold scores', scores_manual, cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_bc, y_bc, cv=cv5))

<details>
<summary><b>💡 Hint</b></summary>

`for tr, va in splitter.split(X): mu, sd = X[tr].mean(0), X[tr].std(0) ...`

</details>

<details>
<summary><b>✅ Solution</b></summary>

A pipeline is exactly this loop: every learned transformation (scaler, imputer, selector, encoder) is refit on the training fold only, preventing the leak
warned about in the note. Report mean ± std across folds, e.g. ≈ 0.97 ± 0.02.

```python
X_bc, y_bc = load_breast_cancer(return_X_y=True)
cv5 = KFold(5, shuffle=True, random_state=0)

def cv_score(X, y, splitter):
    out = []
    for tr, va in splitter.split(X):
        mu, sd = X[tr].mean(0), X[tr].std(0)
        m = LogisticRegression(max_iter=1000).fit((X[tr] - mu) / sd, y[tr])
        out.append(m.score((X[va] - mu) / sd, y[va]))
    return out

scores_manual = cv_score(X_bc, y_bc, cv5)
print(np.mean(scores_manual).round(4), np.std(scores_manual).round(4))
```

</details>

### Exercise 14 · Grid search from scratch
*💻 Code · ★★☆*

On a harder version of `load_diabetes` (150 samples, the 10 real features plus 40 pure-noise features), choose ridge's `alpha` from `np.logspace(-3, 3, 13)` by 5-fold CV (`KFold(5)`) minimising mean validation MSE.
Store the CV curve in `cv_mse` (one value per alpha) and the winner in `best_alpha`. Compare with `GridSearchCV`.

In [ ]:
X_db, y_db = load_diabetes(return_X_y=True)
X_db, y_db = np.c_[X_db[:150], np.random.default_rng(0).normal(size=(150, 40)) * X_db.std()], y_db[:150]   # 150 samples, 10 real + 40 noise features
alphas = np.logspace(-3, 3, 13)
cv_mse = None
best_alpha = None

_gs = GridSearchCV(Ridge(), {'alpha': alphas}, cv=KFold(5), scoring='neg_mean_squared_error').fit(X_db, y_db)
check('CV curve', cv_mse, -_gs.cv_results_['mean_test_score'])
check('best alpha', best_alpha, _gs.best_params_['alpha'])

<details>
<summary><b>💡 Hint</b></summary>

Two nested loops: alphas × folds. `np.argmin` of the mean MSE.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The curve is U-shaped in $\log\alpha$ (minimum at $\alpha = 0.1$): too little regularization overfits the 40 noise features, too much underfits ([[Bias-Variance Tradeoff]]). The selected $\alpha$ is
then used to refit on all training data. Its CV score is slightly optimistic (see the nested-CV exercise).

```python
X_db, y_db = load_diabetes(return_X_y=True)
X_db, y_db = np.c_[X_db[:150], np.random.default_rng(0).normal(size=(150, 40)) * X_db.std()], y_db[:150]   # 150 samples, 10 real + 40 noise features
alphas = np.logspace(-3, 3, 13)
cv_mse = []
for a in alphas:
    errs = [np.mean((Ridge(alpha=a).fit(X_db[tr], y_db[tr]).predict(X_db[va]) - y_db[va]) ** 2)
            for tr, va in KFold(5).split(X_db)]
    cv_mse.append(np.mean(errs))
cv_mse = np.array(cv_mse)
best_alpha = alphas[np.argmin(cv_mse)]
print(best_alpha, cv_mse.round(0))
```

</details>

### Exercise 15 · The optimism of non-nested CV
*💻 Code · ★★★*

On **random labels** (no signal at all), tune $k$ of k-NN over $1..25$ with an inner `StratifiedKFold(5, shuffle=True, random_state=0)`.
(a) Store `GridSearchCV(...).best_score_` in `score_nonnested`. (b) Store the nested estimate (outer `StratifiedKFold(5, shuffle=True, random_state=1)`) in
`score_nested`. Which one tells the truth?

In [ ]:
_r = np.random.default_rng(2)
X_rand = _r.normal(size=(60, 10)); y_rand = _r.integers(0, 2, 60)
inner = StratifiedKFold(5, shuffle=True, random_state=0)
outer = StratifiedKFold(5, shuffle=True, random_state=1)
grid = {'n_neighbors': list(range(1, 26))}
score_nonnested = None
score_nested = None

if score_nonnested is not None:
    check('non-nested is optimistic', score_nonnested > score_nested + 0.1, True)
    check('nested is near chance', abs(score_nested - 0.5) < 0.1, True)
else:
    check('nested vs non-nested', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Nested: `cross_val_score(GridSearchCV(KNeighborsClassifier(), grid, cv=inner), X_rand, y_rand, cv=outer).mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Non-nested ≈ 0.65 on pure noise: picking the best of 25 noisy CV scores selects lucky noise. Nested CV ≈ 0.50 is the honest answer.
With small datasets and many hyperparameters this optimism can be large (Cawley & Talbot, 2010).

```python
_r = np.random.default_rng(2)
X_rand = _r.normal(size=(60, 10)); y_rand = _r.integers(0, 2, 60)
inner = StratifiedKFold(5, shuffle=True, random_state=0)
outer = StratifiedKFold(5, shuffle=True, random_state=1)
grid = {'n_neighbors': list(range(1, 26))}
score_nonnested = GridSearchCV(KNeighborsClassifier(), grid, cv=inner).fit(X_rand, y_rand).best_score_
score_nested = cross_val_score(GridSearchCV(KNeighborsClassifier(), grid, cv=inner), X_rand, y_rand, cv=outer).mean()
print(score_nonnested, score_nested)
```

</details>

### Exercise 16 · Group leakage
*💻 Code · ★★★*

50 patients, 10 recordings each. Each patient has a characteristic feature "fingerprint" and a single diagnosis, but the diagnosis is **unrelated** to the
fingerprint. Compare 1-NN accuracy under shuffled `KFold(5)` (`acc_kfold`) and `GroupKFold(5)` (`acc_group`). Explain the difference.

In [ ]:
_r = np.random.default_rng(5)
P = 50
fingerprint = _r.normal(size=(P, 5)) * 3
diagnosis = _r.integers(0, 2, P)
groups = np.repeat(np.arange(P), 10)
X_pat = fingerprint[groups] + _r.normal(size=(500, 5))
y_pat = diagnosis[groups]
acc_kfold = None
acc_group = None

if acc_kfold is not None:
    check('random k-fold looks great', acc_kfold > 0.8, True)
    check('group k-fold ~ chance', abs(acc_group - 0.5) < 0.15, True)
else:
    check('group leakage', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=GroupKFold(5), groups=groups)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With random folds the nearest neighbour of a recording is almost always another recording **of the same patient**, so the model "predicts" the diagnosis by
recognising the patient: ≈ 0.88. With patients held out entirely it is at chance (≈ 0.55 here; 0.5 up to noise), the true performance on new patients.

```python
_r = np.random.default_rng(5)
P = 50
fingerprint = _r.normal(size=(P, 5)) * 3
diagnosis = _r.integers(0, 2, P)
groups = np.repeat(np.arange(P), 10)
X_pat = fingerprint[groups] + _r.normal(size=(500, 5))
y_pat = diagnosis[groups]
acc_kfold = cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=KFold(5, shuffle=True, random_state=0)).mean()
acc_group = cross_val_score(KNeighborsClassifier(1), X_pat, y_pat, cv=GroupKFold(5), groups=groups).mean()
print(acc_kfold, acc_group)
```

</details>

### Exercise 17 · Time-series split from scratch
*💻 Code · ★★☆*

Implement `ts_split(N, n_splits)` reproducing `TimeSeriesSplit(n_splits)`: test blocks of size `N // (n_splits + 1)` placed at the end, each training set is
everything **before** its test block (expanding window). Test on $N=20$, 4 splits.

In [ ]:
def ts_split(N, n_splits):
    return None   # list of (train_idx, test_idx)

ts_20 = ts_split(20, 4)

_ref = list(TimeSeriesSplit(4).split(np.zeros(20)))
check('time-series folds', None if ts_20 is None else all(np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1]) for a, b in zip(ts_20, _ref)) and len(ts_20) == 4, True)

<details>
<summary><b>💡 Hint</b></summary>

Test size $t = 20 // 5 = 4$; the first test block starts at $N - n_{splits}\,t = 4$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Folds: train $[0,4)$ test $[4,8)$; train $[0,8)$ test $[8,12)$; … Training always precedes testing, mimicking deployment. Shuffled k-fold on time series lets the model
interpolate between future and past observations, overestimating forecast accuracy.

```python
def ts_split(N, n_splits):
    t = N // (n_splits + 1)
    starts = [N - (n_splits - i) * t for i in range(n_splits)]
    return [(np.arange(s), np.arange(s, s + t)) for s in starts]

ts_20 = ts_split(20, 4)
```

</details>

### Exercise 18 · Choosing a polynomial degree: AIC, BIC and CV
*💻 Code · ★★★*

Data: $y = 1 + x - 2x^2 + 0.5x^3 + \varepsilon$, $\varepsilon\sim\mathcal N(0, 0.5^2)$, $x\sim U(-2,2)$, $N=60$. For degrees $1..8$ fit least squares and compute the Gaussian
maximum log-likelihood $\ln\hat L = -\frac N2\big(\ln(2\pi\hat\sigma^2) + 1\big)$ with $\hat\sigma^2 = \text{RSS}/N$, then AIC and BIC with $k = \text{degree} + 2$
(coefficients incl. intercept, plus $\sigma^2$). Also compute 5-fold CV MSE. Store the arrays `aic`, `bic`, `cv` (index 0 = degree 1).

In [ ]:
_rp = np.random.default_rng(1)
x_p = _rp.uniform(-2, 2, 60)
y_p = 1 + x_p - 2 * x_p**2 + 0.5 * x_p**3 + 0.5 * _rp.normal(size=60)
degrees = range(1, 9)
aic = None
bic = None
cv = None

_ll = []
for d in degrees:
    _V = np.vander(x_p, d + 1); _w = np.linalg.lstsq(_V, y_p, rcond=None)[0]; _res = y_p - _V @ _w
    _ll.append(stats.norm.logpdf(_res, scale=np.sqrt(np.mean(_res ** 2))).sum())
_k = np.arange(1, 9) + 2
check('AIC', aic, 2 * _k - 2 * np.array(_ll), tol=1e-4)
check('BIC', bic, _k * np.log(60) - 2 * np.array(_ll), tol=1e-4)
if cv is not None:
    check('CV and BIC both pick degree 3', (int(np.argmin(cv)) + 1, int(np.argmin(bic)) + 1), (3, 3))
else:
    check('CV and BIC both pick degree 3', None, (3, 3))

<details>
<summary><b>💡 Hint</b></summary>

`np.vander(x, d + 1)` gives the design matrix; for CV use `KFold(5, shuffle=True, random_state=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All three criteria drop sharply up to degree 3 and then flatten; BIC and CV choose degree 3 (the truth), while AIC here picks degree 6 because its penalty
per parameter (2) is smaller than BIC's ($\ln60\approx4.1$). With other random seeds the choices vary (CV is itself noisy on 60 points!).
Information criteria needed one fit per degree, CV five.

```python
_rp = np.random.default_rng(1)
x_p = _rp.uniform(-2, 2, 60)
y_p = 1 + x_p - 2 * x_p**2 + 0.5 * x_p**3 + 0.5 * _rp.normal(size=60)
degrees = range(1, 9)
aic, bic, cv = [], [], []
for d in degrees:
    V = np.vander(x_p, d + 1)
    w = np.linalg.lstsq(V, y_p, rcond=None)[0]
    s2 = np.mean((y_p - V @ w) ** 2)
    ll = -60 / 2 * (np.log(2 * np.pi * s2) + 1)
    k = d + 2
    aic.append(2 * k - 2 * ll); bic.append(k * np.log(60) - 2 * ll)
    errs = []
    for tr, va in KFold(5, shuffle=True, random_state=0).split(x_p):
        wt = np.linalg.lstsq(V[tr], y_p[tr], rcond=None)[0]
        errs.append(np.mean((V[va] @ wt - y_p[va]) ** 2))
    cv.append(np.mean(errs))
aic, bic, cv = map(np.array, (aic, bic, cv))
print('AIC picks', np.argmin(aic) + 1, '| BIC picks', np.argmin(bic) + 1, '| CV picks', np.argmin(cv) + 1)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Cross-Validation and Model Selection](Cross-Validation%20and%20Model%20Selection.md).*